<center style="padding: 1em 2em;  margin: 1em 0;">
    <h1 style="font-size: 2.2em; margin: 0 0 0.5em 0; color:rgb(52, 72, 92);">
        «Pandas»
    </h1>
</center>

Сегодня познакомимся с pandas – библиотекой для работы с таблицами. Представьте таблицу участников: рядом стоят имя, возраст и город. Как хранить эти разные данные вместе, сохраняя возраст числом, и затем находить нужные строки? Начнём с этого вопроса.

Pandas часто опирается на NumPy при хранении и обработке числовых столбцов. Мы начнём с одного столбца, соберём таблицу, научимся выбирать строки, считать показатели по группам и соединять данные из двух таблиц. Эти приёмы пригодятся на следующем занятии, когда перейдём к визуализации.

<img src="../../Other/images/pandas.gif" width="560" style="max-width: 100%; height: auto;" alt="Анимация с пандами">

<small>(GIF из <a href="https://miptstats.github.io/courses/python/08_pandas1.html">материалов MIPT Stats</a>)</small>

---

## **План урока**

1. **Столбец и таблица** – `Series`, `DataFrame` и CSV.
2. **Осмотр и пропуски** – размер, типы, первые строки, пустые значения.
3. **Индекс и выборка** – `loc`, `iloc`, маски и изменение ячейки.
4. **Сводка и группировка** – частоты, статистики и `groupby`.
5. **Порядок строк** – сортировка по значениям и меткам.
6. **Соединение** – `merge`, `.join()` и связь с `LEFT JOIN`.

---

## **Столбец и таблица.**

**Зачем ещё одна библиотека?** Возьмём массив NumPy, где в каждой строке стоят имя и возраст. Он может хранить их, но выберет общий тип: число превратится в текст.

In [1]:
import numpy as np

mixed = np.array([["Анна", 25], ["Борис", 30]])
mixed

array([['Анна', '25'],
       ['Борис', '30']], dtype='<U21')

Кавычки вокруг `25` в результате показывают, что это уже строка. Проверим тип всего массива.

In [2]:
mixed.dtype

dtype('<U21')

В таблице удобнее хранить возраст числом, а имя текстом. Начнём с одного столбца. У каждой строки есть **подпись**: например, имя человека. В pandas такую подпись называют **меткой**, а набор меток – **индексом** (`index`). Это позволяет обращаться к строке по имени, а не только по номеру её места.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong><code>Series</code>:</strong><br>Один столбец значений с индексом и именем (<code>name</code>). Метки индекса помогают находить и сопоставлять значения.
</div>

In [3]:
import pandas as pd

ages = pd.Series(
    [25, 30, 35, 40],
    index=["Анна", "Борис", "Виктор", "Галина"],
    name="Возраст",
)
ages

Анна      25
Борис     30
Виктор    35
Галина    40
Name: Возраст, dtype: int64

Под рядом pandas показывает `dtype`. Значения этого числового `Series` можно передать библиотеке, которая ждёт NumPy-массив: `to_numpy()` возвращает их без меток.

In [4]:
ages.to_numpy()

array([25, 30, 35, 40])

Теперь соберём несколько столбцов. В словаре ключ станет именем столбца, а список – его значениями сверху вниз.

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #64748b; background-color: #f8fafc; color: #475569;">
<strong>DataFrame:</strong><br>Двумерная таблица с метками строк и столбцов. При выборе одного столбца мы получаем <code>Series</code>; у разных столбцов могут быть разные типы.
</div>

In [5]:
people = pd.DataFrame({
    "Имя": ["Анна", "Борис", "Виктор", "Галина", "Денис"],
    "Возраст": [25, 30, 35, 28, 22],
    "Зарплата": [80_000, 95_000, 75_000, 85_000, 70_000],
    "Город": ["Москва", "СПб", "Москва", "СПб", "Казань"],
})
people

,Имя,Возраст,Зарплата,Город
0,Анна,25,80000,Москва
1,Борис,30,95000,СПб
2,Виктор,35,75000,Москва
3,Галина,28,85000,СПб
4,Денис,22,70000,Казань


В `ages` метками строк были имена; для `people` мы их не задавали, поэтому pandas поставил слева числа от 0 до 4. Достанем один именованный столбец. Одна пара квадратных скобок возвращает `Series`.

In [6]:
people["Возраст"]

0    25
1    30
2    35
3    28
4    22
Name: Возраст, dtype: int64

Для нескольких столбцов передадим список имён. Даже список из одного имени вернул бы `DataFrame`: здесь мы выбираем таблицу.

In [7]:
people[["Имя", "Возраст"]]

,Имя,Возраст
0,Анна,25
1,Борис,30
2,Виктор,35
3,Галина,28
4,Денис,22


Таблицы часто приходят из файла CSV: первая строка задаёт имена столбцов, следующие – записи. Чтобы пример не зависел от внешнего файла, дадим `read_csv` текст через `StringIO`: для pandas это объект, который можно читать как файл.

In [8]:
from io import StringIO

csv_text = """Имя,Возраст,Город
Анна,25,Москва
Борис,30,СПб
"""
from_text = pd.read_csv(StringIO(csv_text))
from_text

,Имя,Возраст,Город
0,Анна,25,Москва
1,Борис,30,СПб


Pandas распознал возраст как число, а имя как текст. Дальше будем работать с таблицей `people`, где больше строк для сравнения.

---

## **Осмотр таблицы.**

**Сколько данных пришло?** `shape` возвращает две длины: число строк и число столбцов, как у двумерного массива NumPy.

In [9]:
people.shape

(5, 4)

`head(2)` показывает первые две строки. На большом файле это быстрый способ увидеть названия столбцов и вид значений.

In [10]:
people.head(2)

,Имя,Возраст,Зарплата,Город
0,Анна,25,80000,Москва
1,Борис,30,95000,СПб


`tail(2)` смотрит на конец таблицы. Там иногда обнаруживаются записи, которых не было видно в начале.

In [11]:
people.tail(2)

,Имя,Возраст,Зарплата,Город
3,Галина,28,85000,СПб
4,Денис,22,70000,Казань


Если файл упорядочен, первые строки могут быть непохожи на остальные. `sample(2)` берёт случайные строки; `random_state` закрепляет выбор для повторного запуска примера.

In [12]:
people.sample(2, random_state=0)

,Имя,Возраст,Зарплата,Город
2,Виктор,35,75000,Москва
0,Анна,25,80000,Москва


`dtypes` показывает тип каждого столбца. Это полезно проверить до вычислений: возраст должен остаться числом.

In [13]:
people.dtypes

Имя           str
Возраст     int64
Зарплата    int64
Город         str
dtype: object

`columns` хранит имена столбцов, а `index` – метки строк. Начнём с имён.

In [14]:
people.columns

Index(['Имя', 'Возраст', 'Зарплата', 'Город'], dtype='str')

Метки строк здесь созданы автоматически. Позже заменим их именами людей.

In [15]:
people.index

RangeIndex(start=0, stop=5, step=1)

Мы отдельно посмотрели размер таблицы и типы столбцов. `info()` собирает их в одном отчёте и добавляет число заполненных значений: теперь можно проверить знакомые сведения сразу для всех столбцов.

In [16]:
people.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Имя       5 non-null      str  
 1   Возраст   5 non-null      int64
 2   Зарплата  5 non-null      int64
 3   Город     5 non-null      str  
dtypes: int64(2), str(2)
memory usage: 292.0 bytes


### **Пропущенное значение.**

Отложим `people` на минуту: в коротком ряду легче увидеть каждое пустое место. Не каждая запись заполнена. Сделаем короткий ряд баллов с одним пустым местом. `None` в таком числовом ряду pandas распознает как пропуск; здесь значения становятся вещественными, а пропуск отображается как `NaN`.

In [17]:
scores = pd.Series([10.0, None, 30.0], name="Баллы")
scores

0    10.0
1     NaN
2    30.0
Name: Баллы, dtype: float64

`isna()` ставит `True` там, где значения нет. Получается знакомая по NumPy логическая маска.

In [18]:
scores.isna()

0    False
1     True
2    False
Name: Баллы, dtype: bool

Логические значения можно сложить: `True` засчитается как 1. Так посчитаем пустые места.

In [19]:
scores.isna().sum()

np.int64(1)

Среднее по умолчанию пропускает пустое место. Здесь остаются только 10 и 30.

In [20]:
scores.mean()

np.float64(20.0)

Ноль – настоящее значение, а пропуск означает, что значения нет. `fillna(0)` создаёт ряд с нулём вместо пропуска; его среднее будет другим. Исходный `scores` от этого не меняется.

In [21]:
scores.fillna(0).mean()

np.float64(13.333333333333334)

### **Метки участвуют в вычислениях.**

Это ещё один короткий опыт перед возвращением к `people`. У `Series` важны не только значения, но и метки. Подготовим два ряда: во втором строки стоят в другом порядке и отсутствует метка `C`.

In [22]:
base = pd.Series([10, 20, 30], index=["A", "B", "C"])
base

A    10
B    20
C    30
dtype: int64

Во втором ряду порядок меток обратный, а метки `C` нет. Пока просто посмотрим на него.

In [23]:
extra = pd.Series([2, 1], index=["B", "A"])
extra

B    2
A    1
dtype: int64

При сложении pandas сопоставит `A` с `A`, `B` с `B`. Для `C` пары нет, поэтому получится пропуск. Числа результата станут вещественными из-за `NaN`.

In [24]:
base + extra

A    11.0
B    22.0
C     NaN
dtype: float64

Если нужна арифметика по позициям без меток, сначала можно получить NumPy-массив. Но тогда соответствие строк придётся проверять самому. Вернёмся к таблице `people` и научимся выбирать её строки.

---

## **Индекс и выборка.**

**Как взять одну ячейку?** У таблицы есть метки строк и имена столбцов. Пока метки `people` – числа `0…4`, поэтому их легко спутать с номерами позиций. Сначала сделаем столбец «Имя» индексом. `set_index` вернёт новую таблицу; исходная `people` не изменится.

In [25]:
named = people.set_index("Имя")
named

,Возраст,Зарплата,Город
Имя,,,
Анна,25,80000,Москва
Борис,30,95000,СПб
Виктор,35,75000,Москва
Галина,28,85000,СПб
Денис,22,70000,Казань


`loc[метка_строки, имя_столбца]` ищет **метки**. Здесь берём возраст строки с именем «Виктор».

In [26]:
named.loc["Виктор", "Возраст"]

np.int64(35)

`iloc[номер_строки, номер_столбца]` ищет **позиции** от нуля. Виктор стоит третьим, возраст – в первом столбце таблицы `named`.

In [27]:
named.iloc[2, 0]

np.int64(35)

В `loc` можно передать списки меток по обеим осям: выберем двух людей и два названных столбца.

In [28]:
named.loc[["Анна", "Виктор"], ["Возраст", "Город"]]

,Возраст,Город
Имя,,
Анна,25,Москва
Виктор,35,Москва


Обычный срез `iloc` работает как срез списка: правый край не входит. Поэтому `0:2` оставляет две первые строки.

In [29]:
named.iloc[0:2]

,Возраст,Зарплата,Город
Имя,,,
Анна,25,80000,Москва
Борис,30,95000,СПб


У среза `loc` по меткам обе границы входят, если они есть в индексе. Здесь берём строки от Анны до Виктора в текущем порядке.

In [30]:
named.loc["Анна":"Виктор"]

,Возраст,Зарплата,Город
Имя,,,
Анна,25,80000,Москва
Борис,30,95000,СПб
Виктор,35,75000,Москва


<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #3b82f6; background-color: #eff6ff; color: #1e40af;">
<strong>ℹ️ Важно:</strong><br><code>loc</code> читает метки и включает конечную метку в срез. <code>iloc</code> читает позиции и не включает правую границу среза.
</div>

### **Условие выбирает строки.**

Сравнение всего столбца с числом возвращает маску: по одному `True` или `False` на строку. Это тот же первый шаг, что был у нас в NumPy.

In [31]:
older_mask = people["Возраст"] > 30
older_mask

0    False
1    False
2     True
3    False
4    False
Name: Возраст, dtype: bool

Передадим маску таблице. Останутся только строки, где условие истинно.

In [32]:
people[older_mask]

,Имя,Возраст,Зарплата,Город
2,Виктор,35,75000,Москва


Если нужны только некоторые столбцы, `loc` позволяет указать маску для строк и список имён для столбцов в одних скобках.

In [33]:
people.loc[older_mask, ["Имя", "Город"]]

,Имя,Город
2,Виктор,Москва


В SQL похожую задачу записали бы как `SELECT Имя, Город FROM people WHERE Возраст > 30`. В pandas список имён отвечает за столбцы, а маска – за строки. Аналогия помогает читать запрос, хотя у pandas дополнительно есть индекс с метками.

Два условия соединяются через `&`. Каждое сравнение берём в скобки, чтобы Python вычислил его до объединения масок.

In [34]:
young_and_paid = (people["Зарплата"] > 80_000) & (people["Возраст"] < 35)
young_and_paid

0    False
1     True
2    False
3     True
4    False
dtype: bool

Теперь той же маской оставим подходящие записи.

In [35]:
people.loc[young_and_paid, ["Имя", "Возраст", "Зарплата"]]

,Имя,Возраст,Зарплата
1,Борис,30,95000
3,Галина,28,85000


Когда нужно проверить принадлежность нескольким готовым значениям, `isin` тоже создаёт маску. Здесь выберем жителей Москвы и Казани.

In [36]:
in_cities = people["Город"].isin(["Москва", "Казань"])
in_cities

0     True
1    False
2     True
3    False
4     True
Name: Город, dtype: bool

Применим её к таблице.

In [37]:
people.loc[in_cities, ["Имя", "Город"]]

,Имя,Город
0,Анна,Москва
2,Виктор,Москва
4,Денис,Казань


### **Как изменить ячейку.**

Чтение через `loc` мы уже видели. Запись устроена так же: строка, столбец, новое значение. Сделаем копию, чтобы исходная таблица осталась пригодной для следующих опытов.

In [38]:
updated = people.copy()
updated.loc[0, "Зарплата"] = 90_000
updated.loc[0, ["Имя", "Зарплата"]]

Имя          Анна
Зарплата    90000
Name: 0, dtype: object

В исходной таблице зарплата Анны осталась прежней.

In [39]:
people.loc[0, "Зарплата"]

np.int64(80000)

<div style="padding: 12px 16px; margin: 16px 0; border-radius: 6px; border-left: 4px solid #f59e0b; background-color: #fffbf0; color: #92400e;">
<strong>⚠️ Предупреждение:</strong><br>Чтобы изменить исходную таблицу, обращайтесь к ней одним действием через <code>loc</code>. Цепочка вроде <code>df["Зарплата"][0] = 123</code> сначала выбирает отдельный объект; в pandas 3 она не меняет исходную таблицу.
</div>

---

## **Сводка и группировка.**

**Сколько людей в каждом городе?** `value_counts()` считает частоту каждого значения столбца.

In [40]:
people["Город"].value_counts()

Город
Москва    2
СПб       2
Казань    1
Name: count, dtype: int64

`nunique()` отвечает на другой вопрос: сколько разных городов встретилось.

In [41]:
people["Город"].nunique()

3

Числовой столбец можно свести к одному числу. Например, средний возраст считается по всем строкам.

In [42]:
people["Возраст"].mean()

np.float64(28.0)

Медиана – середина упорядоченных возрастов. Её удобно сравнивать со средним, особенно когда встречаются крайние значения.

In [43]:
people["Возраст"].median()

np.float64(28.0)

`agg` (от aggregation, «сведение») задаёт несколько таких вопросов одному столбцу и собирает ответы рядом.

In [44]:
people["Возраст"].agg(["min", "mean", "max"])

min     22.0
mean    28.0
max     35.0
Name: Возраст, dtype: float64

`describe()` собирает числовые сводки в одном отчёте. `count`, `mean`, `min` и `max` уже знакомы. `std` показывает разброс, а `25%`, `50%` и `75%` – значения, ниже которых лежит соответствующая доля наблюдений. На пяти строках это лишь учебный обзор.

In [45]:
people.describe()

,Возраст,Зарплата
count,5.000000,5.000000
mean,28.000000,81000.000000
std,4.949747,9617.692031
min,22.000000,70000.000000
25%,25.000000,75000.000000
50%,28.000000,80000.000000
75%,30.000000,85000.000000
max,35.000000,95000.000000


Позже мы сможем нарисовать возраст и зарплату как точки. Уже сейчас можно численно проверить, есть ли между ними линейная связь: для этого служит `corr`. Она измеряет линейную связь: значения около 1 означают совместный рост, около -1 – противоположное движение, около 0 – слабую линейную связь. Это описание данных, а не доказательство причины.

У нас только пять наблюдений, поэтому само число здесь служит примером метода, а не надёжным выводом о связи возраста и зарплаты.

In [46]:
people[["Возраст", "Зарплата"]].corr().round(2)

,Возраст,Зарплата
Возраст,1.00,0.29
Зарплата,0.29,1.00


### **Одно число на группу.**

Средний возраст всей таблицы ответил на один вопрос. Теперь спросим отдельно по городам. `groupby("Город")` делит строки на группы, выбор `"Возраст"` оставляет нужный столбец, а `mean()` считает среднее внутри каждой группы.

In [47]:
people.groupby("Город")["Возраст"].mean()

Город
Казань    22.0
Москва    30.0
СПб       29.0
Name: Возраст, dtype: float64

В SQL это похоже на `GROUP BY Город` вместе с `AVG(Возраст)`. Слева в результате pandas стоят названия групп: они стали метками индекса.

`agg` может задать две функции внутри каждой группы. `count` считает заполненные зарплаты, `mean` – их среднее. Здесь пропусков нет, поэтому `count` совпадает с числом людей.

In [48]:
people.groupby("Город")["Зарплата"].agg(["count", "mean"])

,count,mean
Город,,
Казань,1,70000.0
Москва,2,77500.0
СПб,2,90000.0


Так одна и та же схема работает для всей таблицы и для каждой группы: сначала выбрать данные, затем свести их к ответу.

---

## **Сортировка.**

**Как поменять порядок строк?** `sort_values("Возраст")` возвращает таблицу от младшего к старшему. Исходная `people` остаётся в прежнем порядке.

In [49]:
people.sort_values("Возраст")

,Имя,Возраст,Зарплата,Город
4,Денис,22,70000,Казань
0,Анна,25,80000,Москва
3,Галина,28,85000,СПб
1,Борис,30,95000,СПб
2,Виктор,35,75000,Москва


`ascending=False` меняет направление. В SQL это соответствует `ORDER BY Зарплата DESC`.

In [50]:
people.sort_values("Зарплата", ascending=False)

,Имя,Возраст,Зарплата,Город
1,Борис,30,95000,СПб
3,Галина,28,85000,СПб
0,Анна,25,80000,Москва
2,Виктор,35,75000,Москва
4,Денис,22,70000,Казань


Если передать несколько столбцов, второй разрешает совпадения по первому. Здесь сначала идут города по алфавиту, а внутри города – зарплаты по убыванию.

In [51]:
people.sort_values(["Город", "Зарплата"], ascending=[True, False])

,Имя,Возраст,Зарплата,Город
4,Денис,22,70000,Казань
0,Анна,25,80000,Москва
2,Виктор,35,75000,Москва
1,Борис,30,95000,СПб
3,Галина,28,85000,СПб


`sort_index()` сортирует **метки**, а не значения столбца. В `named` метками служат имена людей.

In [52]:
named.sort_index()

,Возраст,Зарплата,Город
Имя,,,
Анна,25,80000,Москва
Борис,30,95000,СПб
Виктор,35,75000,Москва
Галина,28,85000,СПб
Денис,22,70000,Казань


После сортировки по возрасту позиции изменятся, а старые числовые метки строк сохранятся. Это наглядная разница между `iloc` и `loc`.

In [53]:
by_age = people.sort_values("Возраст")
by_age[["Имя", "Возраст"]]

,Имя,Возраст
4,Денис,22
0,Анна,25
3,Галина,28
1,Борис,30
2,Виктор,35


Первая **позиция** теперь у Дениса.

In [54]:
by_age.iloc[0, 0]

'Денис'

Метка `0` по-прежнему принадлежит Анне.

In [55]:
by_age.loc[0, "Имя"]

'Анна'

## **Соединение таблиц.**

Одна таблица хранит людей и города, другая – справочник регионов. Чтобы не вписывать регион каждому человеку вручную, сопоставим таблицы по городу. В SQL это задача `LEFT JOIN`; в pandas посмотрим два близких способа её решить.

In [56]:
city_regions = pd.DataFrame({
    "Город": ["Москва", "СПб"],
    "Регион": ["Центр", "Северо-Запад"],
})
city_regions

,Город,Регион
0,Москва,Центр
1,СПб,Северо-Запад


Начнём с `merge`: в обеих таблицах город записан обычным столбцом. `on="Город"` указывает общий ключ, а `how="left"` сохраняет всех людей. Для Казани соответствия в справочнике нет.

In [57]:
joined = people.merge(city_regions, on="Город", how="left")
joined[["Имя", "Город", "Регион"]]

,Имя,Город,Регион
0,Анна,Москва,Центр
1,Борис,СПб,Северо-Запад
2,Виктор,Москва,Центр
3,Галина,СПб,Северо-Запад
4,Денис,Казань,NaN


У Дениса регион остался пустым. Строк по-прежнему пять: каждому городу в справочнике соответствует не больше одной строки. При повторяющихся ключах справа число строк результата могло бы вырасти.

Теперь тот же вопрос решим через `.join()`. Этот метод особенно удобен, когда ключ справа уже стоит в индексе. Перенесём названия городов из столбца справочника в его индекс.

In [58]:
regions_by_city = city_regions.set_index("Город")
regions_by_city

,Регион
Город,
Москва,Центр
СПб,Северо-Запад


В `.join()` параметр `on="Город"` берёт город из столбца `people` и ищет такое же имя в индексе справочника. `how="left"` снова оставляет все строки слева.

In [59]:
people.join(regions_by_city, on="Город", how="left")[["Имя", "Город", "Регион"]]

,Имя,Город,Регион
0,Анна,Москва,Центр
1,Борис,СПб,Северо-Запад
2,Виктор,Москва,Центр
3,Галина,СПб,Северо-Запад
4,Денис,Казань,NaN


Оба способа дали один результат. В нашем примере `merge` сопоставил **два столбца**, а `.join()` – **столбец слева и индекс справа**. У `.join()` левое соединение выбрано по умолчанию.

На языке SQL общий смысл такой: `FROM people LEFT JOIN city_regions ON people.Город = city_regions.Город`. Перед соединением стоит проверить, уникальны ли ключи справа. Есть и отличие от SQL: pandas может сопоставлять пустые ключи друг с другом.

---

## **Дополнительно почитать.**

- [Урок MIPT Stats о pandas](https://miptstats.github.io/courses/python/08_pandas1.html) – примеры с `Series` и `DataFrame`.
- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) – быстрый проход по `Series` и `DataFrame`.
- [Indexing and selecting data](https://pandas.pydata.org/docs/user_guide/indexing.html) – правила `loc`, `iloc` и масок.
- [Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html) – типы пропусков и работа с ними.
- [Group by: split, apply, combine](https://pandas.pydata.org/docs/user_guide/groupby.html) – что делает группировка.
- [Comparison with SQL](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html) – соответствия запросов и выражений pandas.
- [Merge, join, concatenate](https://pandas.pydata.org/docs/user_guide/merging.html) – виды соединений и повторяющиеся ключи.
- [Copy-on-Write](https://pandas.pydata.org/docs/user_guide/migration.html#copy-on-write-cow) – почему изменение выбранного столбца ведёт себя иначе в новых версиях.

Дальше ту же таблицу будем рисовать: на следующей паре посмотрим, как показать распределение и сравнить группы на графике.